# P05 European Climate Trends
## Notebook 02 - Analysis

**Project:** P05 European Climate Trends  
**Author:** Ose Omokhua 

---

## Purpose

This notebook performs the core statistical analysis for P05. It has five jobs:

1. Load the annual temperature and precipitation aggregations produced in
   notebook 01
2. Compute temperature and precipitation anomalies relative to the
   1991-2020 WMO reference period for each country
3. Fit OLS linear trends to each country's annual temperature and
   precipitation series, with Newey-West autocorrelation-robust standard
   errors where serial correlation is detected
4. Apply the Mann-Kendall non-parametric trend test and Sen's slope
   estimator to each country and variable
5. Compile a production-grade results table - one row per country per
   variable - reporting OLS slope, Sen's slope, Mann-Kendall Z statistic,
   p-value, and statistical significance

All analytical functions are imported from `src/analysis.py`. The mathematics
underpinning each method was derived in full at the start of this project
before any code was written.

## 1. Environment Setup

We import all required libraries, load the project configuration, and
verify the working directory. The processed CSV files produced in notebook
01 are loaded here as the primary inputs to all analysis in this notebook.

In [1]:
import os
import sys
import importlib

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
import config
importlib.reload(config)

from src.analysis import ols_trend, mann_kendall_trend, trend_summary, compute_anomaly

print("Working directory:", os.getcwd())
print("Project root     :", config.PROJECT_ROOT)
print()
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("Statsmodels:", sm.__version__)
print()
print("Files in data/processed/:")
for f in sorted(os.listdir(config.DATA_PROCESSED)):
    size_kb = os.path.getsize(os.path.join(config.DATA_PROCESSED, f)) / 1024
    print(f"  {f}  --  {size_kb:.1f} KB")

Working directory: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\notebooks
Project root     : C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5

NumPy      : 2.4.4
Pandas     : 3.0.2
Statsmodels: 0.14.6

Files in data/processed/:
  rr_annual.csv  --  70.8 KB
  rr_quality_report.csv  --  64.7 KB
  tg_annual.csv  --  71.9 KB
  tg_quality_report.csv  --  64.7 KB


## 2. Load Processed Data

We load the four CSV files produced in notebook 01. The annual temperature
and precipitation DataFrames are the primary inputs to all analysis. The
quality reports are loaded for reference and to confirm the zero-failure
result carries forward into this notebook.

After loading we verify shapes, data types, and value ranges to confirm
the files loaded correctly before any analysis begins.

In [2]:
tg_annual_df     = pd.read_csv(os.path.join(config.DATA_PROCESSED, "tg_annual.csv"))
rr_annual_df     = pd.read_csv(os.path.join(config.DATA_PROCESSED, "rr_annual.csv"))
tg_quality_df    = pd.read_csv(os.path.join(config.DATA_PROCESSED, "tg_quality_report.csv"))
rr_quality_df    = pd.read_csv(os.path.join(config.DATA_PROCESSED, "rr_quality_report.csv"))

print("=== Shapes ===")
print(f"  tg_annual_df  : {tg_annual_df.shape}")
print(f"  rr_annual_df  : {rr_annual_df.shape}")
print(f"  tg_quality_df : {tg_quality_df.shape}")
print(f"  rr_quality_df : {rr_quality_df.shape}")
print()
print("=== Data Types ===")
print(tg_annual_df.dtypes)
print()
print("=== TG Annual Preview ===")
print(tg_annual_df.head(5).to_string(index=False))
print()
print("=== RR Annual Preview ===")
print(rr_annual_df.head(5).to_string(index=False))
print()
print("=== Countries in dataset ===")
print(sorted(tg_annual_df["country"].unique()))
print(f"Total: {tg_annual_df['country'].nunique()} countries")
print()
print("=== Year range ===")
print(f"  First year: {tg_annual_df['year'].min()}")
print(f"  Last year : {tg_annual_df['year'].max()}")

=== Shapes ===
  tg_annual_df  : (2250, 3)
  rr_annual_df  : (2250, 3)
  tg_quality_df : (2250, 6)
  rr_quality_df : (2250, 6)

=== Data Types ===
country            str
year             int64
mean_temp_c    float64
dtype: object

=== TG Annual Preview ===
country  year  mean_temp_c
Austria  1950     6.972445
Austria  1951     7.060195
Austria  1952     6.040322
Austria  1953     6.844952
Austria  1954     5.601598

=== RR Annual Preview ===
country  year  precip_mm_yr
Austria  1950    947.275000
Austria  1951    956.094336
Austria  1952   1070.117480
Austria  1953    821.545020
Austria  1954   1154.949414

=== Countries in dataset ===
['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom']
Total: 30 co

## 3. Temperature and Precipitation Anomalies

An anomaly is the departure of a value from a long-term reference mean:

    a_t = y_t - y_bar_ref

where `y_bar_ref` is the mean of the variable over the WMO standard
reference period 1991-2020.

Anomalies serve two purposes in this analysis. First, they make
cross-country comparisons meaningful -- a country with a long-term mean
temperature of 2°C and a country with a long-term mean of 15°C are not
directly comparable in absolute terms, but their anomaly series can be
compared directly because both are expressed as departures from their own
baseline. Second, anomalies communicate climate change clearly to
non-technical audiences -- a statement like "2024 was 1.8°C above the
1991-2020 average" is immediately interpretable without any background
in climatology.

We compute anomalies for both temperature and precipitation for all 30
countries. The reference period mean is computed from the subset of each
country's annual series that falls within 1991-2020, then subtracted from
the full 1950-2024 series.

In [3]:
tg_anomaly_records = []
rr_anomaly_records = []

for country in config.COUNTRIES:
    tg_country = tg_annual_df[tg_annual_df["country"] == country].set_index("year")["mean_temp_c"]
    rr_country = rr_annual_df[rr_annual_df["country"] == country].set_index("year")["precip_mm_yr"]

    tg_anomaly = compute_anomaly(tg_country, config.REFERENCE_PERIOD_START, config.REFERENCE_PERIOD_END)
    rr_anomaly = compute_anomaly(rr_country, config.REFERENCE_PERIOD_START, config.REFERENCE_PERIOD_END)

    for year in tg_anomaly.index:
        tg_anomaly_records.append({
            "country":      country,
            "year":         year,
            "tg_anomaly_c": tg_anomaly.loc[year],
        })
    for year in rr_anomaly.index:
        rr_anomaly_records.append({
            "country":      country,
            "year":         year,
            "rr_anomaly_mm": rr_anomaly.loc[year],
        })

tg_anomaly_df = pd.DataFrame(tg_anomaly_records)
rr_anomaly_df = pd.DataFrame(rr_anomaly_records)

print("=== TG Anomaly DataFrame ===")
print("Shape:", tg_anomaly_df.shape)
print(tg_anomaly_df.head(10).to_string(index=False))
print()
print("=== RR Anomaly DataFrame ===")
print("Shape:", rr_anomaly_df.shape)
print(rr_anomaly_df.head(10).to_string(index=False))
print()
print("=== Anomaly sanity checks ===")
print("TG anomaly mean across all country-years (should be close to 0 for ref period):")
ref_tg = tg_anomaly_df[
    tg_anomaly_df["year"].between(config.REFERENCE_PERIOD_START, config.REFERENCE_PERIOD_END)
]["tg_anomaly_c"].mean()
print(f"  Mean TG anomaly 1991-2020: {ref_tg:.6f} C")
print()
print("TG anomaly mean 2015-2024 (should be positive -- recent warming):")
recent_tg = tg_anomaly_df[
    tg_anomaly_df["year"] >= 2015
]["tg_anomaly_c"].mean()
print(f"  Mean TG anomaly 2015-2024: {recent_tg:.3f} C")

=== TG Anomaly DataFrame ===
Shape: (2250, 3)
country  year  tg_anomaly_c
Austria  1950     -0.495622
Austria  1951     -0.407872
Austria  1952     -1.427745
Austria  1953     -0.623115
Austria  1954     -1.866469
Austria  1955     -1.697795
Austria  1956     -2.414397
Austria  1957     -0.764718
Austria  1958     -0.984770
Austria  1959     -0.632043

=== RR Anomaly DataFrame ===
Shape: (2250, 3)
country  year  rr_anomaly_mm
Austria  1950     -78.855013
Austria  1951     -70.035677
Austria  1952      43.987467
Austria  1953    -204.584993
Austria  1954     128.819401
Austria  1955     -15.375716
Austria  1956      -6.474349
Austria  1957     -28.801302
Austria  1958     122.614909
Austria  1959     -31.636849

=== Anomaly sanity checks ===
TG anomaly mean across all country-years (should be close to 0 for ref period):
  Mean TG anomaly 1991-2020: 0.000000 C

TG anomaly mean 2015-2024 (should be positive -- recent warming):
  Mean TG anomaly 2015-2024: 0.751 C


## 4. Trend Analysis

We now apply the three-method trend detection pipeline derived at the start
of this project to every country and both variables.

For each country and variable we compute:

- **OLS linear trend** -- slope in units per decade with 95% confidence
  interval, R-squared, and p-value
- **Mann-Kendall test** -- non-parametric test for monotonic trend,
  returning the Z statistic, p-value, and trend direction
- **Sen's slope** -- robust median-of-pairwise-slopes estimator in units
  per decade

The three methods are combined into a single results row per country per
variable using the `trend_summary()` function from `src/analysis.py`.

A finding is considered robust when all three of the following conditions
are met:

1. Mann-Kendall p-value is below 0.05
2. OLS and Sen's slope agree in sign
3. The OLS 95% confidence interval excludes zero

Results are compiled into a single DataFrame with one row per country per
variable, then saved to `data/processed/` for use in notebook 03.

In [4]:
trend_records = []

for country in config.COUNTRIES:
    tg_country = tg_annual_df[
        tg_annual_df["country"] == country
    ].set_index("year")["mean_temp_c"]

    rr_country = rr_annual_df[
        rr_annual_df["country"] == country
    ].set_index("year")["precip_mm_yr"]

    tg_summary = trend_summary(tg_country, country, "temperature")
    rr_summary = trend_summary(rr_country, country, "precipitation")

    trend_records.append(tg_summary)
    trend_records.append(rr_summary)

trends_df = pd.DataFrame(trend_records)

print("=== Trend Results DataFrame ===")
print("Shape:", trends_df.shape)
print("Columns:", list(trends_df.columns))
print()
print("=== Temperature Trends -- top 10 fastest warming countries ===")
tg_trends = trends_df[trends_df["variable"] == "temperature"].copy()
tg_trends = tg_trends.sort_values("sens_decade", ascending=False)
print(tg_trends[[
    "country", "ols_slope_decade", "sens_decade",
    "mk_z", "mk_p_value", "significant"
]].head(10).to_string(index=False))
print()
print("=== Precipitation Trends -- significant trends only ===")
rr_trends = trends_df[trends_df["variable"] == "precipitation"].copy()
rr_sig = rr_trends[rr_trends["significant"] == True].sort_values("sens_decade")
print(rr_sig[[
    "country", "ols_slope_decade", "sens_decade",
    "mk_z", "mk_p_value", "significant"
]].to_string(index=False))
print()
print("=== Summary counts ===")
print(f"Temperature trends significant (p < 0.05): "
      f"{tg_trends['significant'].sum()} of {len(tg_trends)}")
print(f"Precipitation trends significant (p < 0.05): "
      f"{rr_trends['significant'].sum()} of {len(rr_trends)}")

=== Trend Results DataFrame ===
Shape: (60, 12)
Columns: ['country', 'variable', 'ols_slope_decade', 'ols_ci_lower', 'ols_ci_upper', 'ols_r_squared', 'ols_p_value', 'mk_z', 'mk_p_value', 'mk_trend', 'sens_decade', 'significant']

=== Temperature Trends -- top 10 fastest warming countries ===
  country  ols_slope_decade  sens_decade     mk_z   mk_p_value  significant
  Estonia          0.347903     0.353036 6.166146 6.997447e-10         True
  Ukraine          0.353036     0.343456 6.440604 1.189993e-10         True
   Latvia          0.343321     0.341534 6.147849 7.854082e-10         True
  Belgium          0.315721     0.335428 6.504644 7.787770e-11         True
  Finland          0.306635     0.328593 5.168950 2.354125e-07         True
  Austria          0.314201     0.328517 6.742507 1.556777e-11         True
 Slovakia          0.313097     0.316778 6.303375 2.912335e-10         True
  Czechia          0.302369     0.313121 6.092957 1.108436e-09         True
  Germany          0.29

## 5. Export Trend Results

We save the full trend results DataFrame to `data/processed/`. This file
contains one row per country per variable -- 60 rows total -- with all
OLS, Mann-Kendall, and Sen's slope outputs. It is the primary input to
notebook 03 where production figures are generated.

We also save the anomaly DataFrames which are needed for the time series
visualisations in notebook 03.

In [5]:
trends_path      = os.path.join(config.DATA_PROCESSED, "trends.csv")
tg_anomaly_path  = os.path.join(config.DATA_PROCESSED, "tg_anomaly.csv")
rr_anomaly_path  = os.path.join(config.DATA_PROCESSED, "rr_anomaly.csv")

trends_df.to_csv(trends_path, index=False)
tg_anomaly_df.to_csv(tg_anomaly_path, index=False)
rr_anomaly_df.to_csv(rr_anomaly_path, index=False)

print("Saved:")
print(f"  trends.csv      -- {trends_df.shape[0]} rows, {trends_df.shape[1]} columns")
print(f"  tg_anomaly.csv  -- {tg_anomaly_df.shape[0]} rows, {tg_anomaly_df.shape[1]} columns")
print(f"  rr_anomaly.csv  -- {rr_anomaly_df.shape[0]} rows, {rr_anomaly_df.shape[1]} columns")
print()
print("Files in data/processed/:")
for f in sorted(os.listdir(config.DATA_PROCESSED)):
    size_kb = os.path.getsize(os.path.join(config.DATA_PROCESSED, f)) / 1024
    print(f"  {f}  --  {size_kb:.1f} KB")

Saved:
  trends.csv      -- 60 rows, 12 columns
  tg_anomaly.csv  -- 2250 rows, 3 columns
  rr_anomaly.csv  -- 2250 rows, 3 columns

Files in data/processed/:
  rr_annual.csv  --  70.8 KB
  rr_anomaly.csv  --  73.3 KB
  rr_quality_report.csv  --  64.7 KB
  tg_annual.csv  --  71.9 KB
  tg_anomaly.csv  --  75.2 KB
  tg_quality_report.csv  --  64.7 KB
  trends.csv  --  11.5 KB


## 6. Full Results Table

We produce a clean, readable summary of all trend results. This table is
the analytical core of P05 -- it quantifies warming and precipitation
change rates for every country with full statistical attribution.

The table reports:

- OLS slope per decade with 95% confidence interval
- Sen's slope per decade as the robust cross-check
- Mann-Kendall Z statistic and p-value
- Whether the trend is statistically significant at p < 0.05
- Whether OLS and Sen's slope agree in sign -- a robustness flag

A finding is marked as fully robust when it is significant AND OLS and
Sen's slope agree in sign. This is the standard we committed to at the
start of this project -- a finding must survive both parametric and
non-parametric scrutiny before we report it as a conclusion.

In [6]:
results_table = trends_df.copy()

# Add robustness flag -- True when significant AND OLS and Sen's slope agree in sign
results_table["signs_agree"] = (
    np.sign(results_table["ols_slope_decade"]) == np.sign(results_table["sens_decade"])
)
results_table["robust"] = results_table["significant"] & results_table["signs_agree"]

# Add formatted CI string for readability
results_table["ols_ci_95"] = results_table.apply(
    lambda row: f"[{row['ols_ci_lower']:.3f}, {row['ols_ci_upper']:.3f}]", axis=1
)

# Separate temperature and precipitation
tg_table = results_table[results_table["variable"] == "temperature"].copy()
rr_table = results_table[results_table["variable"] == "precipitation"].copy()

# Sort both by Sen's slope descending
tg_table = tg_table.sort_values("sens_decade", ascending=False).reset_index(drop=True)
rr_table = rr_table.sort_values("sens_decade", ascending=False).reset_index(drop=True)

print("=" * 80)
print("TEMPERATURE TRENDS -- OLS and Sen's Slope per Decade (degrees C)")
print("=" * 80)
print(tg_table[[
    "country", "ols_slope_decade", "sens_decade",
    "ols_ci_95", "mk_z", "mk_p_value", "robust"
]].to_string(index=False))

print()
print("=" * 80)
print("PRECIPITATION TRENDS -- OLS and Sen's Slope per Decade (mm)")
print("=" * 80)
print(rr_table[[
    "country", "ols_slope_decade", "sens_decade",
    "ols_ci_95", "mk_z", "mk_p_value", "robust"
]].to_string(index=False))

print()
print("=== Robustness Summary ===")
print(f"Temperature -- robust trends: {tg_table['robust'].sum()} of {len(tg_table)}")
print(f"Precipitation -- robust trends: {rr_table['robust'].sum()} of {len(rr_table)}")
print()
print("=== Non-robust temperature trends (if any) ===")
tg_not_robust = tg_table[tg_table["robust"] == False]
if len(tg_not_robust) == 0:
    print("  None -- all temperature trends are robust")
else:
    print(tg_not_robust[["country", "ols_slope_decade", "sens_decade", "mk_p_value"]].to_string(index=False))
print()
print("=== Non-significant precipitation trends ===")
rr_not_sig = rr_table[rr_table["significant"] == False]
print(f"  {len(rr_not_sig)} countries with non-significant precipitation trends")
print(rr_not_sig[["country", "ols_slope_decade", "sens_decade", "mk_p_value"]].to_string(index=False))

TEMPERATURE TRENDS -- OLS and Sen's Slope per Decade (degrees C)
       country  ols_slope_decade  sens_decade      ols_ci_95     mk_z   mk_p_value  robust
       Estonia          0.347903     0.353036 [0.258, 0.437] 6.166146 6.997447e-10    True
       Ukraine          0.353036     0.343456 [0.264, 0.442] 6.440604 1.189993e-10    True
        Latvia          0.343321     0.341534 [0.256, 0.431] 6.147849 7.854082e-10    True
       Belgium          0.315721     0.335428 [0.245, 0.387] 6.504644 7.787770e-11    True
       Finland          0.306635     0.328593 [0.203, 0.410] 5.168950 2.354125e-07    True
       Austria          0.314201     0.328517 [0.246, 0.382] 6.742507 1.556777e-11    True
      Slovakia          0.313097     0.316778 [0.237, 0.389] 6.303375 2.912335e-10    True
       Czechia          0.302369     0.313121 [0.227, 0.377] 6.092957 1.108436e-09    True
       Germany          0.297414     0.311838 [0.226, 0.369] 6.458901 1.054663e-10    True
     Lithuania          0

## 7. Export Final Results Table

We save the full results table including robustness flags to
`data/processed/`. This is the definitive analytical output of notebook 02
and the primary reference for the policy brief and README findings tables.

In [7]:
final_results_path = os.path.join(config.DATA_PROCESSED, "trends_final.csv")

results_table.to_csv(final_results_path, index=False)

print("Saved: trends_final.csv")
print(f"  Rows    : {results_table.shape[0]}")
print(f"  Columns : {results_table.shape[1]}")
print(f"  Columns : {list(results_table.columns)}")
print()
print("Files in data/processed/:")
for f in sorted(os.listdir(config.DATA_PROCESSED)):
    size_kb = os.path.getsize(os.path.join(config.DATA_PROCESSED, f)) / 1024
    print(f"  {f}  --  {size_kb:.1f} KB")

Saved: trends_final.csv
  Rows    : 60
  Columns : 15
  Columns : ['country', 'variable', 'ols_slope_decade', 'ols_ci_lower', 'ols_ci_upper', 'ols_r_squared', 'ols_p_value', 'mk_z', 'mk_p_value', 'mk_trend', 'sens_decade', 'significant', 'signs_agree', 'robust', 'ols_ci_95']

Files in data/processed/:
  rr_annual.csv  --  70.8 KB
  rr_anomaly.csv  --  73.3 KB
  rr_quality_report.csv  --  64.7 KB
  tg_annual.csv  --  71.9 KB
  tg_anomaly.csv  --  75.2 KB
  tg_quality_report.csv  --  64.7 KB
  trends.csv  --  11.5 KB
  trends_final.csv  --  13.2 KB
